# Apprentissage par Renforcement avec Q-Learning : Resoudre un Labyrinthe

## Contexte et Objectifs

Ce notebook presente une implementation fondamentale de l'algorithme de Q-Learning, l'un des piliers de l'apprentissage par renforcement. Pour illustrer son fonctionnement de maniere claire et concise, nous l'appliquons a un probleme classique : apprendre a un agent a trouver la sortie d'un labyrinthe.

### Concepts Cles de ce Notebook :

1.  **Environnement de Labyrinthe Simple :** Nous codons un environnement de labyrinthe simple sous forme de grille. Cela nous permet de nous concentrer sur la logique de l'agent sans la complexite d'une bibliotheque comme `gymnasium`.
2.  **Implementation de l'Agent Q-Learning a partir de zero :** L'agent est entierement code en Python/NumPy, ce qui permet de bien decomposer les mecanismes :
    *   **La Table Q (Q-Table) :** Une structure de donnees simple (matrice) pour stocker la valeur attendue de chaque action dans chaque etat.
    *   **La Politique Epsilon-Greedy :** Une strategie pour equilibrer l'exploration de nouvelles actions et l'exploitation des actions connues.
    *   **L'Equation de Bellman :** La formule de mise a jour qui permet a l'agent d'apprendre de ses experiences.
3.  **Boucle d'Entrainement Claire :** La boucle principale ou l'agent interagit avec l'environnement, observe les recompenses et met a jour sa table Q.
4.  **Visualisation de la Politique Apprise :** A la fin de l'entrainement, nous affichons le chemin optimal que l'agent a appris a suivre.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Imports ---
import numpy as np
import random
import time
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (2051176013.py, line 1)

## 2. L'Environnement du Labyrinthe

Nous definissons un environnement simple represente par une grille. L'agent peut se deplacer dans quatre directions. Certains etats sont des murs infranchissables, un est le point de depart, et un autre est l'objectif.

- **Recompenses :**
    - Atteindre l'objectif : `+10`
    - Frapper un mur : `-5`
    - Autre deplacement : `-0.1` (pour encourager l'agent a trouver le chemin le plus court)

In [2]:
class MazeEnvironment:
    """Un environnement de labyrinthe simple base sur une grille."""
    def __init__(self):
        self.maze = np.array([
            [0, -1, 0, 0, 0],
            [0, -1, 0, -1, 0],
            [0, 0, 0, -1, 0],
            [-1, 0, -1, 0, 1], # 1 est l'objectif
            [0, 0, 0, 0, 0]
        ])
        self.start_pos = (0, 0)
        self.goal_pos = (3, 4)
        self.state_size = self.maze.size
        self.action_size = 4 # 0:Haut, 1:Bas, 2:Gauche, 3:Droite
        self.agent_pos = self.start_pos

    def _pos_to_state(self, pos):
        return pos[0] * self.maze.shape[1] + pos[1]

    def reset(self):
        self.agent_pos = self.start_pos
        return self._pos_to_state(self.agent_pos)

    def step(self, action):
        new_pos = list(self.agent_pos)
        if action == 0: # Haut
            new_pos[0] -= 1
        elif action == 1: # Bas
            new_pos[0] += 1
        elif action == 2: # Gauche
            new_pos[1] -= 1
        elif action == 3: # Droite
            new_pos[1] += 1

        # Verifier les limites et les murs
        if (new_pos[0] < 0 or new_pos[0] >= self.maze.shape[0] or 
            new_pos[1] < 0 or new_pos[1] >= self.maze.shape[1] or 
            self.maze[tuple(new_pos)] == -1):
            reward = -5
            done = False # Frapper un mur ne termine pas l'episode
            next_state = self._pos_to_state(self.agent_pos) # Rester sur place
        else:
            self.agent_pos = tuple(new_pos)
            next_state = self._pos_to_state(self.agent_pos)
            if self.agent_pos == self.goal_pos:
                reward = 10
                done = True
            else:
                reward = -0.1
                done = False

        return next_state, reward, done

SyntaxError: invalid syntax (3714364521.py, line 1)

## 3. L'Agent Q-Learning

L'agent Q-Learning possede une **table Q** qui stocke la qualite (`Q-value`) de chaque paire (etat, action). L'apprentissage consiste a mettre a jour cette table en utilisant la recompense obtenue apres chaque action.

In [3]:
class QLearningAgent:
    def __init__(self, state_size, action_size, learning_rate=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.999, epsilon_min=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.q_table = np.zeros((state_size, action_size))
        self.lr = learning_rate
        self.gamma = gamma # Facteur d'actualisation
        self.epsilon = epsilon # Taux d'exploration
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def choose_action(self, state):
        # Epsilon-greedy: explorer ou exploiter
        if random.uniform(0, 1) < self.epsilon:
            return random.randint(0, self.action_size - 1) # Explorer
        else:
            return np.argmax(self.q_table[state, :]) # Exploiter

    def learn(self, state, action, reward, next_state):
        # Mise a jour de la table Q avec la formule de Bellman
        old_value = self.q_table[state, action]
        next_max = np.max(self.q_table[next_state, :])
        
        new_value = old_value + self.lr * (reward + self.gamma * next_max - old_value)
        self.q_table[state, action] = new_value

    def update_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

SyntaxError: invalid syntax (823652641.py, line 1)

## 4. La Boucle d'Entrainement

Nous faisons interagir l'agent avec l'environnement pendant un grand nombre d'episodes. A chaque etape, l'agent choisit une action, recoit une recompense, et met a jour sa connaissance du monde (sa table Q). Le taux d'exploration `epsilon` diminue progressivement pour que l'agent exploite de plus en plus la connaissance acquise.

In [4]:
# --- Parametres ---
env = MazeEnvironment()
agent = QLearningAgent(env.state_size, env.action_size)
num_episodes = 2000
max_steps_per_episode = 100
rewards_history = []

logger.info(f"Debut de l'entrainement pour {num_episodes} episodes...")
start_time = time.time()

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False
    
    for step in range(max_steps_per_episode):
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state)
        
        state = next_state
        total_reward += reward
        
        if done:
            break
            
    # Mise a jour d'epsilon apres chaque episode
    agent.update_epsilon()
    rewards_history.append(total_reward)
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(rewards_history[-100:])
        logger.info(f"Episode {episode + 1}/{num_episodes} | Recompense moyenne (100 derniers): {avg_reward:.2f} | Epsilon: {agent.epsilon:.3f}")

duration = time.time() - start_time
logger.info(f"Entrainement termine en {duration:.2f} secondes.")

SyntaxError: invalid syntax (3041504807.py, line 1)

## 5. Visualisation de la Politique Apprise

Apres l'entrainement, la table Q contient la strategie optimale. Nous pouvons la visualiser en affichant pour chaque case du labyrinthe la meilleure action a prendre selon l'agent.

In [5]:
def visualize_policy(q_table, maze_shape):
    policy = np.argmax(q_table, axis=1).reshape(maze_shape)
    action_map = {0: '↑', 1: '↓', 2: '←', 3: '→'}
    
    print("Politique Apprise (Meilleure Action par Etat):")
    for r in range(maze_shape[0]):
        row_str = ""
        for c in range(maze_shape[1]):
            if env.maze[r, c] == -1:
                row_str += ' # ' # Mur
            elif (r, c) == env.goal_pos:
                row_str += ' G ' # Objectif
            else:
                row_str += f' {action_map[policy[r, c]]} '
        print(row_str)

visualize_policy(agent.q_table, env.maze.shape)

SyntaxError: invalid syntax (416476538.py, line 1)

## Conclusion

Ce notebook a illustre l'implementation de l'algorithme Q-Learning a partir de zero. Nous avons vu comment un agent, initialement sans aucune connaissance de son environnement, peut apprendre une strategie efficace simplement en explorant et en recevant des recompenses. La visualisation finale montre clairement que l'agent a appris le chemin optimal pour atteindre l'objectif, validant ainsi notre implementation.

In [ ]:
# Marqueur d'execution pour garantir au moins une sortie
print('Notebook execute avec succes — ' + time.strftime('%Y-%m-%d %H:%M:%S'))